In [25]:
import pandas as pd
import networkx as nx
import numpy as np
from tqdm import tqdm
import folium
import sqlite3
import os
import wget

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [26]:
def haversine_np(lon1, lat1, lon2, lat2, factor=1.25):
    """Calculate the great circle distance between two points
    on the earth (specified in decimal degrees).

    All args must be of equal length.
    """
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2

    c = 2 * np.arcsin(np.sqrt(a))
    km = 6367 * c
    # 1.25 because the road distance is, on average, 25% larger than a straight flight
    return factor * km

In [27]:
url = 'https://hitchmap.com/dump.sqlite'
filename = 'dump.sqlite'
if os.path.exists(filename):
        os.remove(filename)
filename = wget.download(url)

points = pd.read_sql('select * from points', sqlite3.connect(filename))
df = points[~points["dest_lon"].isna()]
df.head(1)

,id,lat,lon,rating,country,wait,nickname,comment,datetime,reviewed,banned,ip,dest_lat,dest_lon,signal,ride_datetime,user_id,from_hitchwiki
18195,18207,49.635165,5.970091,4.0,LU,NaN,None,Got a ride after 10 minutes withhout even bothering straight to Lure close to Switzerland which was where I needed to go. Lovely,2017-09-03 20:40:31.000000,1,0,,47.6864,6.4943,None,None,NaN,NaN


In [28]:
MAX_WALKING_DISTANCE = 10 # in km

min_lon = 3
max_lon = 15
min_lat = 50
max_lat = 54
df = df[df["lon"] > min_lon]
df = df[df["lon"] < max_lon]
df = df[df["lat"] < max_lat]
df = df[df["lat"] > min_lat]
df = df[df["dest_lon"] > min_lon]
df = df[df["dest_lon"] < max_lon]
df = df[df["dest_lat"] < max_lat]
df = df[df["dest_lat"] > min_lat]


In [29]:
df["dist"] = df.apply(lambda row: haversine_np(row.lon, row.lat, row.dest_lon, row.dest_lat), axis=1)
# df = df[df["dist"] > 100]

In [30]:
# df = df[df["country"] == "BE"].iloc[0:1]

In [31]:
g = nx.DiGraph()

In [32]:
for _, row in df.iterrows():
    # only using above 100km rides
    if row["dist"]:
        # adding edges is suffient, nodes are created automatically
        g.add_edge(
            (row.lon, row.lat),
            (row.dest_lon, row.dest_lat),
            weight=row["dist"],
        )

In [33]:
g.edges(data=True)

OutEdgeDataView([((4.7322507218876, 51.563595927427), (4.4040077, 50.8188354327487), {'weight': 107.32570463762464}), ((4.7322507218876, 51.563595927427), (4.635797071760664, 51.3588702768895), {'weight': 29.637547312463738}), ((12.4932231903, 51.3541222644), (13.690737962842833, 51.07914802323865), {'weight': 110.97193197827286}), ((12.4932231903, 51.3541222644), (12.708091735839844, 51.25976642202132), {'weight': 22.80185628812606}), ((12.4932231903, 51.3541222644), (13.681411743164062, 51.06185628565291), {'weight': 111.08373792964683}), ((12.4932231903, 51.3541222644), (13.571462631225588, 51.060979488377896), {'weight': 102.28677730687656}), ((4.743854641918098, 51.49984396629321), (5.031051635742188, 51.95484531418707), {'weight': 67.86120437356529}), ((3.8288640975952153, 50.899061033439835), (3.7263393402099614, 51.03869547963934), {'weight': 21.369158585886325}), ((14.49448585510254, 50.149353115812566), (14.08376455307007, 50.480606016557275), {'weight': 58.68928523320751}), 

In [34]:
len(g.nodes)

2340

In [35]:
m = folium.Map([50.7, 4.2], zoom_start=7)

In [36]:
points = list(g.nodes)
for p in points:
    folium.Marker(location=(p[1], p[0])).add_to(m)

for e in g.edges:
    folium.PolyLine(
        [[e[0][1], e[0][0]], [e[1][1], e[1][0]]],
        color="#FF0000",
        weight=5,
    ).add_to(m)

    folium.RegularPolygonMarker(location=[e[1][1], e[1][0]], fill_color='blue', number_of_sides=3, radius=10).add_to(m)

In [37]:
# make it fully connected with walking paths
WALKING_FACTOR = 20 # that is 5 km/h vs. 100 km/h
for n1 in g.nodes:
    for n2 in g.nodes:
        if n1 != n2 and not g.has_edge(n1, n2):
            dist = haversine_np(n1[0], n1[1], n2[0], n2[1])
            if dist < MAX_WALKING_DISTANCE:
                g.add_edge(
                    n1,
                    n2,
                    weight=dist * WALKING_FACTOR,
                )

In [38]:
points = list(g.nodes)
for p in points:
    folium.Marker(location=(p[1], p[0])).add_to(m)

for e in g.edges:
    folium.PolyLine(
        [[e[0][1], e[0][0]], [e[1][1], e[1][0]]],
        color="#FF0000",
        weight=5,
    ).add_to(m)

    folium.RegularPolygonMarker(location=[e[1][1], e[1][0]], fill_color='blue', number_of_sides=3, radius=10).add_to(m)

In [39]:
def find_route(G, departure, arrival):
    num_edges_before = len(G.edges)

    prev_nodes = list(G.nodes)
    for n in prev_nodes:
        dist = haversine_np(arrival[0], arrival[1], n[0], n[1])
        if dist < MAX_WALKING_DISTANCE:
            G.add_edge(
                n,
                arrival,
                weight=dist * WALKING_FACTOR,
            )

    # there is also the walking path from departure to arrival added here
    prev_nodes = list(G.nodes)
    for n in prev_nodes:
        dist = haversine_np(departure[0], departure[1], n[0], n[1])
        if dist < MAX_WALKING_DISTANCE:
            G.add_edge(
                departure,
                n,
                weight=dist * WALKING_FACTOR,
            )

    print(len(G.edges) - num_edges_before, "edges added")

    return (
        G,
        nx.dijkstra_path(G, departure, arrival, weight="weight"),
        nx.dijkstra_path_length(G, departure, arrival, weight="weight"),
    )


In [40]:
# for u, v, data in g.edges(data=True):
#     print(f"Edge from {u} to {v} has weight {data['weight']}")

In [44]:
A = (13.7373, 51.0504) # Antwerp
B = (4.9041, 52.3676) # Amsterdam
g, route, length = find_route(g, A, B)
route, length

50 edges added


([(13.7373, 51.0504),
  (13.7395338714123, 51.0797458759402),
  (13.74269485473633, 51.081224543988796),
  (13.7359946966, 51.099993409),
  (13.489151000976564, 52.36511830615023),
  (13.5137200356, 52.3639013815),
  (13.498158277139256, 52.31810348020336),
  (13.498131765, 52.3181413089),
  (7.43363755010956, 52.318354077002866),
  (7.429675996875506, 52.32621502725702),
  (4.892934097087633, 52.33887682841606),
  (4.9041, 52.3676)],
 np.float64(1195.8141491129854))

In [47]:
m = folium.Map([50.7, 4.2], zoom_start=6)

In [48]:
points = list(g.nodes)
for p in route:
    folium.Marker(location=(p[1], p[0])).add_to(m)

for i, stop in enumerate(route[:-1]):
    folium.PolyLine(
        [[stop[1], stop[0]], [route[i + 1][1], route[i + 1][0]]],
        color="#FF0000",
        weight=5,
    ).add_to(m)

    folium.RegularPolygonMarker(
        location=[route[i + 1][1], route[i + 1][0]], fill_color="blue", number_of_sides=3, radius=10
    ).add_to(m)

m